# Curriculum 01 · Lab 3 — Markdown Structure

**Goal:** Split a markdown document by its header hierarchy — not by raw
character counts — so every chunk carries the section it belongs to.

```
Splitter    : MarkdownHeaderTextSplitter (langchain-text-splitters)
Cuts on     : # / ## / ### headings
Chunk meta  : {"H1": ..., "H2": ..., "H3": ...}
Demos       : Data/local-docs/docs + the repo's own README.md & playbook plan
No API keys : pure LangChain, fully local
```

This lab compares against lab 01 (`RecursiveCharacterTextSplitter`) and lab 02
(`TokenTextSplitter`): those split on **size**, this one splits on **structure**.

**Why this matters for retrieval:** a plain character splitter produces chunks
with no idea of their section, so a question about "persistence" can surface a
chunk from the wrong part of the doc. With header metadata, retrieval can scope
by section — e.g. filter to chunks whose `H1 == "FAISS similarity search"`
before ranking, or show the section path alongside every hit.

See `Topics/Project-04-Markdown-Documentation-RAG/README.md` for the project card.


## 0 · Setup — dependencies & imports

Two things to know before running:

* **`langchain-core` / `langchain-text-splitters`** provide `Document` and
  `MarkdownHeaderTextSplitter` — install cell below if missing.
* **Repo-root paths.** The `.py` resolves `Data/local-docs/...` and
  `README.md` relative to the repo root via `Path(__file__)`. A notebook has
  no `__file__`, and nbclient runs the kernel with its working directory set
  to the notebook's own folder, so the config cell below resolves the repo
  root explicitly — it works when the notebook is executed from the repo
  root, and walks up to find the folder containing `src/splitters/` if you run it
  from anywhere else.

**WHAT TO EXPECT:** The install cell prints pip's progress; the import cell
runs silently. No API keys, no internet, no model downloads.


In [1]:
# Needed for THIS lab only:
#   langchain-core            → Document
#   langchain-text-splitters  → MarkdownHeaderTextSplitter
%pip install langchain-core langchain-text-splitters



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter


## 1 · Config — what to split on, which files

**WHAT:** One splitter configuration (which heading levels become which
metadata keys) and the two document sets: the tiny local docs under
`Data/local-docs/` and the repo's own long markdown files.

**WHY:** `HEADERS_TO_SPLIT_ON` is module-level so the lab is tweakable — add
`("####", "H4")` to split deeper, or drop `("###", "H3")` to merge H3 sections
into their H2 parent. `DOC_FILES` and `LONG_MD_PATHS` stay exactly the
relative names the `.py` prints; only the *base* directory is anchored to the
resolved repo root so the notebook runs wherever the kernel starts.

**WHAT TO EXPECT:** A cell that defines the constants and prints the config
summary — the same three lines the `.py` prints first.


In [3]:
# --- repo-root anchor ---------------------------------------------------
# The .py resolves `Data/local-docs/...` and `README.md` relative to the repo
# root via its ``__file__``. A notebook has no ``__file__`` and nbclient runs
# its kernel with the working directory set to the notebook's own folder, so
# resolve the repo root explicitly: works when launched from the repo root,
# and walks up to the folder containing ``src/splitters/`` if run from elsewhere.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src" / "splitters").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# Module-level constant so the lab is tweakable: add ("####", "H4") to split
# deeper, or drop ("###", "H3") to merge H3 sections into their H2 parent.
HEADERS_TO_SPLIT_ON: list[tuple[str, str]] = [
    ("#", "H1"),
    ("##", "H2"),
    ("###", "H3"),
]

DOCS_DIR = REPO_ROOT / "Data" / "local-docs" / "docs"
DOC_FILES = ["bge-embeddings.md", "faiss-search.md"]

# The repo's own long markdown files (paths relative to the repo root) — the
# primary demo documents with a real #/##/### hierarchy.
LONG_MD_PATHS = ["README.md", ".omo/plans/layer1-rag-playbook.md"]

print(f"Headers to split on: {HEADERS_TO_SPLIT_ON}")
print(f"Local docs: {DOC_FILES}")
print(f"Long repo docs: {LONG_MD_PATHS}\n")


Headers to split on: [('#', 'H1'), ('##', 'H2'), ('###', 'H3')]
Local docs: ['bge-embeddings.md', 'faiss-search.md']
Long repo docs: ['README.md', '.omo/plans/layer1-rag-playbook.md']



## 2 · Helpers — load, split, preview

**WHAT:** Four small functions: read a markdown file as raw text, collect the
distinct `H1` values seen across chunks, split text on its header hierarchy,
and truncate a chunk's content for printing.

**WHY:** Keeping them as named helpers mirrors the `.py` and keeps the demo
cells one-liners — the splitter is built fresh per call so `strip_headers` can
differ between the side-by-side comparisons.

**WHAT TO EXPECT:** Definitions only — no output until the cells below use them.


In [4]:
def load_markdown(path: Path) -> str:
    """Read a markdown file as raw text (no loader needed for plain .md)."""
    return path.read_text(encoding="utf-8")


def distinct_h1_values(chunks: list[Document]) -> list[str]:
    """Distinct H1 values across chunks, in first-appearance order."""
    seen: list[str] = []
    for chunk in chunks:
        h1 = chunk.metadata.get("H1")
        if h1 is not None and h1 not in seen:
            seen.append(h1)
    return seen


def split_markdown(text: str, strip_headers: bool) -> list[Document]:
    """Split markdown text on its header hierarchy.

    Args:
        text: raw markdown content.
        strip_headers: if True the heading line is removed from the chunk
            content and kept only in metadata; if False the heading stays in
            the content too.

    Returns:
        List of Documents whose ``metadata`` carries the header chain
        (e.g. ``{"H1": "FAISS similarity search", "H2": "Persistence"}``).
    """
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=HEADERS_TO_SPLIT_ON,
        strip_headers=strip_headers,
    )
    return splitter.split_text(text)


def preview(text: str, limit: int = 200) -> str:
    """Truncate a chunk's content for printing."""
    return text[:limit] + ("..." if len(text) > limit else "")


## 3 · Load — raw markdown text from disk

**WHAT:** Reads the two local docs (`bge-embeddings.md`, `faiss-search.md`) as
plain text.

**WHY:** Plain `.md` needs no loader — the raw text is exactly what the
splitter consumes, and keeping `docs` as `{filename: text}` lets every later
cell pick a file by name.

**WHAT TO EXPECT:** A silent dict — two keys' worth of text, printed nothing.


In [5]:
docs: dict[str, str] = {
    name: load_markdown(DOCS_DIR / name) for name in DOC_FILES
}


## 4 · Split — structure-preserving, headers kept in content

**WHAT:** Runs `MarkdownHeaderTextSplitter` over both local docs with
`strip_headers=False` (the default), so the heading line stays inside the chunk
and the chunk is self-contained when read or embedded.

**WHY:** This is the "no idea of their section" fix: instead of cutting at
character counts, the splitter cuts at every `#`/`##`/`###` heading and carries
the heading chain into each chunk's `metadata` as `{"H1": ..., "H2": ...,
"H3": ...}`.

**WHAT TO EXPECT:** The chunk totals — one number across both files plus a
per-file breakdown.


In [6]:
chunks_kept: dict[str, list[Document]] = {
    name: split_markdown(text, strip_headers=False)
    for name, text in docs.items()
}
total_kept = sum(len(c) for c in chunks_kept.values())
print(f"strip_headers=False -> {total_kept} chunk(s) across {len(docs)} file(s)")
for name, chunks in chunks_kept.items():
    print(f"  {name}: {len(chunks)} chunk(s)")


strip_headers=False -> 8 chunk(s) across 2 file(s)
  bge-embeddings.md: 4 chunk(s)
  faiss-search.md: 4 chunk(s)


## 5 · Inspect metadata — the header chain each chunk carries

**WHAT:** Looks at the first chunk of `bge-embeddings.md`: its `metadata` and a
200-character preview of its content.

**WHY:** The metadata is the whole point of this splitter — each chunk knows
its `H1`/`H2`/`H3` position in the document, which retrieval can later scope
by.

**WHAT TO EXPECT:** `{'H1': ...}` (plus `H2` where the section is nested) and
the opening lines of the chunk, heading included.


In [7]:
first = chunks_kept[DOC_FILES[0]][0]
print(f"\nFirst chunk of {DOC_FILES[0]} — metadata (header chain):")
print(f"  {first.metadata}")
print(f"  Content preview: {preview(first.page_content)!r}")



First chunk of bge-embeddings.md — metadata (header chain):
  {'H1': 'BGE embeddings'}
  Content preview: '# BGE embeddings  \nBGE (BAAI General Embedding) is a family of open-source embedding models\ntrained by the Beijing Academy of Artificial Intelligence. They convert text\ninto vectors: sentences that me...'


## 6 · The real demo — the repo's own long markdown docs

**WHAT:** Splits `README.md` and `.omo/plans/layer1-rag-playbook.md` — long,
real markdown files with a genuine `#`/`##`/`###` hierarchy — exactly what this
splitter is for.

**WHY:** The tiny local docs are convenient, but the payoff shows on documents
that are actually structured: hundreds of lines of prose and headings turn into
section-aligned chunks, and the distinct-`H1` count shows how many top-level
sections each file contains.

**WHAT TO EXPECT:** A total chunk count across both files, then per file the
chunk count and the number of distinct `H1` values.


In [8]:
long_chunks: dict[str, list[Document]] = {
    path: split_markdown(load_markdown(REPO_ROOT / path), strip_headers=False)
    for path in LONG_MD_PATHS
}
total_long = sum(len(c) for c in long_chunks.values())
print(f"\nLong repo markdown -> {total_long} chunk(s) across {len(LONG_MD_PATHS)} file(s)")
for path, chunks in long_chunks.items():
    h1s = distinct_h1_values(chunks)
    print(f"  {path}: {len(chunks)} chunk(s), {len(h1s)} distinct H1 value(s)")



Long repo markdown -> 23 chunk(s) across 2 file(s)
  README.md: 14 chunk(s), 1 distinct H1 value(s)
  .omo/plans/layer1-rag-playbook.md: 9 chunk(s), 1 distinct H1 value(s)


## 7 · A look inside — README.md's first chunk

**WHAT:** Inspects the first chunk of the README split: its `metadata` chain
and content preview.

**WHY:** The README's very first `#` heading is the document title, so this
chunk shows how the splitter treats a title-level heading — the H1 lands in
metadata, and the intro paragraph stays a self-contained chunk.

**WHAT TO EXPECT:** `{'H1': 'RAG Playbook'}` with the README's opening lines.


In [9]:
readme_chunks = long_chunks["README.md"]
print(f"\nFirst chunk of README.md — metadata (header chain):")
print(f"  {readme_chunks[0].metadata}")
print(f"  Content preview: {preview(readme_chunks[0].page_content)!r}")



First chunk of README.md — metadata (header chain):
  {'H1': 'RAG Playbook'}
  Content preview: '# RAG Playbook  \n> **RAG from Zero to Advanced** — a hands-on, component-swappable curriculum for\n> Retrieval-Augmented Generation, with a benchmark-driven research series on\n> special document format...'


## 8 · Side by side — strip_headers=False vs True

**WHAT:** Splits the same `faiss-search.md` twice — once with the heading kept
inside the content, once stripped — and compares the `## Persistence` section
(chunk index 2 in both cases).

**WHY:** `strip_headers=True` keeps the section path in `metadata` but removes
the heading text from the chunk, which saves tokens and avoids duplicating the
heading in every chunk of that section. `False` (the default) keeps the chunk
self-contained when read or embedded on its own.

**WHAT TO EXPECT:** Identical `metadata` on both sides — the only difference is
whether `## Persistence` itself appears at the start of the content.

(chunk 0 = `# FAISS similarity search` intro, chunk 1 = `## What it does`,
chunk 2 = `## Persistence`, chunk 3 = `## Querying`.)


In [10]:
faiss_text = docs["faiss-search.md"]
kept = split_markdown(faiss_text, strip_headers=False)
stripped = split_markdown(faiss_text, strip_headers=True)
# chunk 0 = "# FAISS similarity search" intro, chunk 1 = "## What it
# does", chunk 2 = "## Persistence", chunk 3 = "## Querying".
print("\nSide by side — '## Persistence' section of faiss-search.md:")
print("  strip_headers=False (heading inside content):")
print(f"    metadata: {kept[2].metadata}")
print(f"    content : {preview(kept[2].page_content)!r}")
print("  strip_headers=True  (heading only in metadata):")
print(f"    metadata: {stripped[2].metadata}")
print(f"    content : {preview(stripped[2].page_content)!r}")



Side by side — '## Persistence' section of faiss-search.md:
  strip_headers=False (heading inside content):
    metadata: {'H1': 'FAISS similarity search', 'H2': 'Persistence'}
    content : '## Persistence  \nFAISS in LangChain is an in-memory index. To keep it between sessions, save and\nload it explicitly:  \nvector_store.save_local("faiss_index/")\nvector_store = FAISS.load_local("faiss_in...'
  strip_headers=True  (heading only in metadata):
    metadata: {'H1': 'FAISS similarity search', 'H2': 'Persistence'}
    content : 'FAISS in LangChain is an in-memory index. To keep it between sessions, save and\nload it explicitly:  \nvector_store.save_local("faiss_index/")\nvector_store = FAISS.load_local("faiss_index/", embeddings...'


## 9 · The payoff — section-scoped retrieval

**WHAT:** Because every chunk carries its H1/H2/H3 chain, a vector store can
filter *before* ranking — shown here as a fictional metadata-filter query.

**WHY:** This is the retrieval benefit of structure-preserving splitting: a
query about persisting a FAISS index can be scoped to `{'H1': 'FAISS similarity
search'}`, so a "persistence" hit can never come from the BGE page. The
metadata the splitter produced in section 4 is what makes that filter possible.

**WHAT TO EXPECT:** A textual illustration of the filter — no vector store or
embeddings are built in this lab.


In [11]:
print("\nSection-scoped retrieval (fictional metadata filter):")
print("  query = 'How do I persist a FAISS index?'")
print("  filter = {'H1': 'FAISS similarity search'}  # scope to one doc")
print("  -> only chunks whose metadata['H1'] matches are embedded/ranked,")
print("     so a 'persistence' hit can never come from the BGE page.")
print("\nTakeaway: structure-preserving splitting carries the section")
print("hierarchy into chunk metadata — retrieval can scope by section.")



Section-scoped retrieval (fictional metadata filter):
  query = 'How do I persist a FAISS index?'
  filter = {'H1': 'FAISS similarity search'}  # scope to one doc
  -> only chunks whose metadata['H1'] matches are embedded/ranked,
     so a 'persistence' hit can never come from the BGE page.

Takeaway: structure-preserving splitting carries the section
hierarchy into chunk metadata — retrieval can scope by section.


## Takeaway

**What you should notice:**

- `MarkdownHeaderTextSplitter` cut both local docs into small, section-aligned
  chunks and the repo's long markdown into dozens of `H1`-scoped chunks.
- Every chunk's `metadata` carries its header chain — `{"H1": ..., "H2": ...,
  "H3": ...}` — which plain character/token splitters never provide.
- `strip_headers` only changes whether the heading text stays in the content;
  the metadata chain is identical either way.
- With header metadata, retrieval can filter by section before ranking —
  scoping a query to one document's H1.

Compare with lab 01 (`RecursiveCharacterTextSplitter`, size-based) and lab 02
(`TokenTextSplitter`, token-based): this lab's splitter is **structure-based**.
